# Fully connected neural network for a sine-to-cosine task

This notebook follows the basic sequence of ideas in deep learning:

1. data
2. forward pass
3. activation functions
4. loss
5. backpropagation
6. training
7. generalization

We will learn a simple regression problem: given a point on a sinusoid and the curve parameters, the network predicts the corresponding cosine value.

In [ ]:
import numpy as np
import torch
from torch import nn
from matplotlib import pyplot as plt

np.random.seed(0)
torch.manual_seed(0)

# Build a dataset where each sample has a different amplitude, frequency, and phase.
# The target is a cosine wave: y = A * cos(f * x + phi)
n_samples = 2000
x = torch.rand(n_samples, 1) * 2.0 - 1.0
A = torch.rand(n_samples, 1) * 1.5 + 0.5
f = torch.rand(n_samples, 1) * 4.0 + 0.5
phi = torch.rand(n_samples, 1) * 2.0 * np.pi

y = A * torch.cos(f * x + phi)
X = torch.cat([x, A, f, phi], dim=1)

print("X shape:", X.shape)
print("y shape:", y.shape)

# Show a few examples from the dataset
fig, ax = plt.subplots(figsize=(8, 4))
for i in range(5):
    xx = torch.linspace(-1.0, 1.0, 500)
    yy = A[i, 0] * torch.cos(f[i, 0] * xx + phi[i, 0])
    ax.plot(xx.numpy(), yy.numpy(), alpha=0.9)
ax.set_title("Examples of the target function")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.show()


## 1) Data: a family of cosine curves

A neural network learns from data, so we first create a dataset in which the label changes with the parameters of the curve:

$$
y = A \cos(fx + \phi)
$$

The network sees the tuple `[x, A, f, phi]` and tries to predict the corresponding value `y`. This makes the problem a regression task, and the samples vary in amplitude, frequency, and phase.

In [ ]:
# First few samples from the dataset
for i in range(3):
    print(f"sample {i}: x={X[i, 0].item():.3f}, A={X[i, 1].item():.3f}, f={X[i, 2].item():.3f}, phi={X[i, 3].item():.3f}, y={y[i, 0].item():.3f}")


## 2) Forward pass: a tiny fully connected network

A fully connected layer performs a linear transformation:

$$
h = Wx + b
$$

Then we apply a nonlinearity. A tiny MLP can be written as:

$$
\hat{y} = W_3\,\sigma(W_2\,\sigma(W_1 x + b_1) + b_2) + b_3
$$

This is the forward pass: input features go through the network, and we obtain a prediction.

In [ ]:
class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 32),
            nn.Tanh(),
            nn.Linear(32, 32),
            nn.Tanh(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x)

model = TinyMLP()
print(model)

with torch.no_grad():
    sample_pred = model(X[:5])
    print("Predictions shape:", sample_pred.shape)
    print(sample_pred[:3])


## 3) Activation functions: why nonlinearity matters

Without activation functions, multiple linear layers would still collapse into a single linear map. That means the model could only fit affine functions, not curved waveforms.

A common choice is the hyperbolic tangent:

$$
	anh(z) = rac{e^z - e^{-z}}{e^z + e^{-z}}
$$

The hidden layers in this example use `tanh`, which is smooth and centered at zero, making it suitable for small regression tasks.

In [ ]:
z = torch.linspace(-3, 3, 400)
t = torch.tanh(z)

plt.figure(figsize=(7, 3))
plt.plot(z.numpy(), t.numpy(), label='tanh')
plt.axhline(0, color='black', lw=0.6, alpha=0.5)
plt.axvline(0, color='black', lw=0.6, alpha=0.5)
plt.title('tanh activation')
plt.xlabel('z')
plt.ylabel('tanh(z)')
plt.legend(frameon=False)
plt.show()


## 4) Loss: measuring prediction error

The model is not told the correct weights directly. Instead, it learns by minimizing a loss function. For regression, mean squared error is a standard choice:

$$
\mathcal{L} = rac{1}{N}\sum_{i=1}^{N}(\hat{y}_i - y_i)^2
$$

This penalizes larger mistakes more strongly and nudges the network toward predictions close to the target.

In [ ]:
criterion = nn.MSELoss()

example_prediction = torch.tensor([0.9, 0.2, -0.5], dtype=torch.float32)
example_target = torch.tensor([1.0, 0.0, -0.4], dtype=torch.float32)
example_loss = criterion(example_prediction, example_target)
print('Example MSE loss:', float(example_loss))


## 5) Backpropagation: learning from the loss

Backpropagation computes the gradient of the loss with respect to every weight and bias. The chain rule lets us propagate the error backward through the network:

$$
rac{\partial \mathcal{L}}{\partial W} \quad	ext{and}\quad rac{\partial \mathcal{L}}{\partial b}
$$

In PyTorch, we can do this automatically with:

```python
loss.backward()
```

Then an optimizer updates the parameters.

In [ ]:
model = TinyMLP()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# One training step
optimizer.zero_grad()
pred = model(X[:32])
loss = criterion(pred.squeeze(-1), y[:32].squeeze(-1))
loss.backward()
optimizer.step()

print('One-step loss:', float(loss))
print('Gradient norm for first layer:', model.net[0].weight.grad.norm().item())


## 6) Training: iterative improvement

Training is just many iterations of the same loop:

1. compute a prediction
2. measure the error
3. backpropagate the error
4. update the weights

This is the heart of learning in neural networks.

In [ ]:
model = TinyMLP()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Split the data into train and validation sets
perm = torch.randperm(len(X))
train_idx = perm[: int(0.8 * len(X))]
val_idx = perm[int(0.8 * len(X)) :]

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]

loss_history = []
for epoch in range(1200):
    model.train()
    optimizer.zero_grad()
    pred = model(X_train)
    loss = criterion(pred.squeeze(-1), y_train.squeeze(-1))
    loss.backward()
    optimizer.step()
    loss_history.append(float(loss))

print(f"Final training loss: {loss_history[-1]:.6f}")

plt.figure(figsize=(8, 3))
plt.plot(loss_history)
plt.title('Training loss over epochs')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.show()


## 7) Generalization: does the model learn the pattern?

A model that only memorizes training examples is not useful. We care whether it learns the underlying rule, not just the specific data points.

We evaluate the trained network on unseen examples and compare its predictions with the true cosine curves.

In [ ]:
model.eval()
with torch.no_grad():
    val_pred = model(X_val).squeeze(-1)
    val_loss = criterion(val_pred, y_val.squeeze(-1))
    print(f"Validation MSE: {float(val_loss):.6f}")

# Visualize one unseen curve
idx = 0
x_plot = torch.linspace(-1.0, 1.0, 500).unsqueeze(1)
A_plot = X_val[idx, 1]
f_plot = X_val[idx, 2]
phi_plot = X_val[idx, 3]

true_curve = A_plot * torch.cos(f_plot * x_plot.squeeze(-1) + phi_plot)
input_vec = torch.cat([
    x_plot,
    torch.full_like(x_plot, A_plot),
    torch.full_like(x_plot, f_plot),
    torch.full_like(x_plot, phi_plot),
], dim=1)
pred_curve = model(input_vec).squeeze(-1)

plt.figure(figsize=(8, 4))
plt.plot(x_plot.squeeze(-1).numpy(), true_curve.numpy(), label='True curve', lw=2)
plt.plot(x_plot.squeeze(-1).numpy(), pred_curve.numpy(), label='Model prediction', lw=2, ls='--')
plt.title('Generalization on an unseen cosine example')
plt.xlabel('x')
plt.ylabel('y')
plt.legend(frameon=False)
plt.show()
